# Case study: missing keywords and invalidation reasons, imputation and univariate outliers

*Session 4, block 2 · Author: course team · Licence: CC-BY-4.0*

**Questions of the practice block**

1. Is a missing keyword list related to the issuing country, the language or the year? What does that say
   about the missing-data mechanism (MCAR, MAR, MNAR)?
2. Why is `invalidation_reason` missing for 85 % of the decisions, and why is that not a problem to impute?
3. How well do simple, KNN and iterative imputation recover the **number of keywords** of a decision when
   we hide it on purpose?
4. Which description lengths and validity durations are outliers by the IQR rule, the z-score and the MAD
   rule?

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

decisions = pd.read_parquet(DATA / "train.parquet")
decisions["keywords_missing"] = decisions["keywords"].isna()
print(f"{len(decisions):,} decisions, keywords missing for {decisions['keywords_missing'].sum():,} "
      f"({decisions['keywords_missing'].mean():.2%})")

## 1. Turn missingness into a variable

A **missingness indicator** is 1 where the value is missing. Comparing it across other variables tells us
whether the rows with and without keywords differ. If the missing share varies with an observed variable,
**MCAR** (missing completely at random) is implausible.

In [ ]:
by_country = (decisions.groupby("issuing_country")["keywords_missing"]
              .agg(share_missing="mean", n_missing="sum", n="size")
              .query("n >= 1000").sort_values("share_missing", ascending=False))
by_country.head(8).round(4)

In [ ]:
by_year = decisions.groupby(decisions["start_date"].dt.year)["keywords_missing"].mean()
ax = by_year.plot.bar(color="#2a78d6", rot=0, figsize=(6, 3))
ax.set(xlabel="start year", ylabel="share without keywords")
ax.axhline(decisions["keywords_missing"].mean(), color="#52514e", ls="--", lw=1)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**A test of independence.** The chi-square test asks whether the missing share is the same in all
countries (null hypothesis: missingness is independent of the country). With 300,000 decisions, even small
differences are significant, so look at the size of the differences as well (applied recap from the
statistics module).

In [ ]:
from scipy.stats import chi2_contingency

big = decisions[decisions["issuing_country"].isin(by_country.index)]
table = pd.crosstab(big["issuing_country"], big["keywords_missing"])
chi2, p_value, dof, _ = chi2_contingency(table)
print(f"chi2 = {chi2:.0f}, dof = {dof}, p = {p_value:.1e}")

**Can the observed columns predict missingness?** If a classifier predicts the indicator better than chance
(area under the ROC curve, AUC, above 0.5), the data are not MCAR. The variables it uses are the ones an
imputation model should include. (Logistic regression and AUC are taught in Sessions 6 and 8; here we only
read the number.)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer

features = pd.DataFrame({
    "issuing_country": decisions["issuing_country"],
    "year": decisions["start_date"].dt.year,
    "log_chars": np.log(decisions["description"].str.len()),
})
model = make_pipeline(
    make_column_transformer((OneHotEncoder(handle_unknown="ignore"), ["issuing_country"]),
                            remainder=StandardScaler()),
    LogisticRegression(max_iter=1000),
)
auc = cross_val_score(model, features, decisions["keywords_missing"], cv=5, scoring="roc_auc")
print(f"AUC for predicting missing keywords: {auc.mean():.2f}")

**Interpretation.** Keywords are missing much more often in decisions from Poland, Slovakia and Austria
than from Germany, and more often in 2017–2018 than later. So the keywords are **not MCAR**. Missingness is
plausibly explained by observed variables (the issuing office and its practice in a given year), which
makes **MAR** a reasonable working assumption; whether a missing keyword list also depends on the product
itself (MNAR) cannot be decided from these data.

**TODO 1.** Write two sentences for the data card: what does this pattern mean if an analysis uses the
English keywords to describe the products of each country?

## 2. Structural missingness: the invalidation reason

`invalidation_reason` is missing for 85 % of the decisions. This is not a data defect: a decision that is
still valid, or that ran its normal three years, has no reason to be invalidated. Imputing a "reason" for
these decisions would invent facts. The right treatment is to keep the value missing and, if needed, derive
a clear variable such as `ended_early`.

In [ ]:
validity_days = (decisions["end_date"] - decisions["start_date"]).dt.days
summary = pd.DataFrame({
    "status": decisions["status"],
    "has_reason": decisions["invalidation_reason"].notna(),
    "full_three_years": validity_days.between(1094, 1096),
})
pd.crosstab([summary["status"], summary["has_reason"]], summary["full_three_years"], margins=True)

Almost all decisions without a reason ran their full three years (1,094–1,096 days), and almost all
decisions with a reason ended earlier. The small off-diagonal groups (274 and 52 decisions) are worth a
look: they are either data errors or special cases, a minor finding for the quality report.

## 3. Compare imputation methods on hidden values

We cannot check an imputation against the truth on the real missing keyword lists. Instead we take the
decisions *with* keywords, compute the **number of keywords**, hide 30 % of these values, impute, and compare
with the hidden values. The predictors are simple features of the description (they are available for every
decision).

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

sample = decisions[decisions["keywords"].notna()].sample(40_000, random_state=2026).reset_index(drop=True)
text = sample["description"]
known = pd.DataFrame({
    "log_chars": np.log(text.str.len()),
    "log1p_digits": np.log1p(text.str.count(r"\d")),
    "n_lines": text.str.count("\n") + 1,
    "is_german": sample["language"].eq("de").astype(int),
    "n_keywords": sample["keywords"].str.split(",").str.len(),
})
rng = np.random.default_rng(2026)
hide = rng.random(len(known)) < 0.3                    # MCAR hiding, for a fair comparison
X = known.copy()
X.loc[hide, "n_keywords"] = np.nan
truth = known.loc[hide, "n_keywords"].to_numpy()
print(known.corr()["n_keywords"].round(2))

imputers = {
    "mean": SimpleImputer(strategy="mean"),
    "median": SimpleImputer(strategy="median"),
    "KNN (k=10, scaled)": make_pipeline(StandardScaler(), KNNImputer(n_neighbors=10)),
    "iterative": IterativeImputer(random_state=0),
}
rows = []
for name, imputer in imputers.items():
    filled = imputer.fit_transform(X)
    if name.startswith("KNN"):                                # undo the scaling for the target column
        scaler = imputer.named_steps["standardscaler"]
        filled = filled * scaler.scale_ + scaler.mean_
    pred = filled[hide, -1]
    rows.append({"method": name, "RMSE (keywords)": np.sqrt(np.mean((pred - truth) ** 2)),
                 "SD of imputed values": pred.std(), "SD of true values": truth.std()})

In [ ]:
# a domain rule: the median number of keywords of the same heading (falls back to the overall median)
heading = sample["heading"]
heading_median = known.loc[~hide, "n_keywords"].groupby(heading[~hide]).median()
pred = heading[hide].map(heading_median).fillna(known.loc[~hide, "n_keywords"].median()).to_numpy()
rows.append({"method": "median of the same heading", "RMSE (keywords)": np.sqrt(np.mean((pred - truth) ** 2)),
             "SD of imputed values": pred.std(), "SD of true values": truth.std()})
pd.DataFrame(rows).round(3)

**Interpretation.** When the course team ran this notebook, the RMSE fell only from 2.29 keywords (mean)
to 2.22 (KNN) and 2.20 (iterative); the median of the same heading did best with 2.15. The description
features are only weakly related to the number of keywords (correlations of 0.24 and below). Two lessons:

- Imputation cannot create information that the other variables do not contain.
- Mean and median imputation give every missing row the same value: the spread of the imputed values is 0,
  far below the true spread. Analyses of variability or correlation become biased.

**TODO 2.** Add `add_indicator=True` to the `SimpleImputer` and look at the extra column. Why is the
indicator useful for a model even when the imputed value itself is poor?

In [ ]:
# TODO 2

## 4. Univariate outliers: IQR rule, z-score and MAD

Three rules flag a value as an outlier:

- **IQR rule (Tukey):** below Q1 − 1.5·IQR or above Q3 + 1.5·IQR;
- **z-score:** |x − mean| / SD > 3;
- **robust z-score (MAD):** 0.6745 · |x − median| / MAD > 3.5, with MAD = median(|x − median|).

In [ ]:
def outlier_flags(x: pd.Series) -> pd.DataFrame:
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    z = (x - x.mean()) / x.std()
    mad = (x - x.median()).abs().median()
    robust_z = 0.6745 * (x - x.median()) / mad if mad > 0 else pd.Series(np.nan, index=x.index)
    return pd.DataFrame({"iqr": (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr),
                         "z": z.abs() > 3,
                         "mad": robust_z.abs() > 3.5})


length = decisions["description"].str.len()
pd.DataFrame({
    "description length": outlier_flags(length).sum(),
    "log description length": outlier_flags(np.log(length)).sum(),
})

In [ ]:
# what are the flagged descriptions on the log scale?
flags = outlier_flags(np.log(length))
flagged = decisions.loc[flags["mad"], ["language", "description", "keywords"]].assign(n_chars=length[flags["mad"]])
flagged.sort_values("n_chars").iloc[[0, 1, 2, -3, -2, -1]]

On the raw scale, the IQR rule flags only long descriptions, because the distribution is right-skewed. On
the log scale, the rules flag both tails: descriptions of a few characters ("TEST", a product name only) and
very long technical descriptions of several thousand characters. The long ones are genuine. Some very short
ones are not real decisions at all: "TEST" (with the keyword TEST) and the Italian "prova" (*test*) look like
test entries that reached the public database. Flag them; a model cannot learn a product from them.

In [ ]:
# validity duration in days: most decisions run exactly three years
validity = validity_days[decisions["end_date"].dt.year > 1900]        # without the placeholder dates
print(validity.describe().round(0))
print("MAD:", (validity - validity.median()).abs().median())
outlier_flags(validity).sum()

**TODO 3.** Why does the MAD rule return no result for the validity duration? Which rule would you use
instead to find decisions with an unusual duration? (Hint: what is the *normal* duration, and what does a
shorter one mean?)

In [ ]:
# TODO 3